# Object-Oriented Programming (OOP) in Python
## A Hands-On Tutorial for Movement Neuroscience Graduate Students

---

**Why this tutorial?**  
As a movement neuroscience researcher, you work with complex, structured data every day — EMG signals from multiple muscles, kinematic trajectories of limb segments, trial metadata, and participant demographics. Object-Oriented Programming (OOP) gives you a powerful way to **organize** all of this into clean, reusable, and scalable code.

This notebook walks you through the core concepts of OOP in Python, using examples drawn from movement neuroscience research. By the end, you'll be able to define your own classes to model experiments, participants, trials, and neural/biomechanical data.

**Prerequisites:** Basic Python (variables, functions, lists, dictionaries).  
**Based on:** [Real Python — OOP in Python 3](https://realpython.com/python3-object-oriented-programming/)

---

## Table of Contents

**Part I — Core Concepts**
1. [What Is Object-Oriented Programming?](#1)
2. [Why OOP Matters in Movement Neuroscience](#2)
3. [Defining a Class in Python](#3)
4. [Instantiating a Class — Creating Objects](#4)
5. [Class vs Instance Attributes](#5)
6. [Instance Methods — Adding Behavior](#6)
7. [Dunder Methods — `__init__` and `__str__`](#7)

**Part II — Inheritance and Polymorphism**
8. [Inheritance — Building Class Hierarchies](#8)
9. [Overriding and Extending Parent Methods](#9)

**Part III — Properties, Encapsulation, and Composition**
10. [The `@property` Decorator — Computed Attributes](#10)
11. [Encapsulation — Protecting Internal Data](#11)
12. [Composition — "Has-A" Relationships](#12)

**Part IV — Dataclasses and Full Framework**
13. [Dataclasses — Less Boilerplate for Data Containers](#13)
14. [Putting It All Together — A Mini Movement Neuroscience Framework](#14)

**Part V — Practice**
15. [Exercises](#15)
16. [Summary & Further Reading](#16)

---
# Part I — Core Concepts

## 1. What Is Object-Oriented Programming? <a id='1'></a>

Object-oriented programming is a **programming paradigm** that structures code by bundling related **data** (properties/attributes) and **behaviors** (methods/functions) into individual **objects**.

Think about how you naturally organize your research:

- A **Participant** has properties (age, handedness, injury history) and behaviors (performs a reaching trial, rests between blocks).
- A **Trial** has properties (start time, condition, target location) and behaviors (records data, computes reaction time).
- An **EMG Signal** has properties (sampling rate, raw voltage data, muscle name) and behaviors (filters the signal, computes RMS).

OOP lets you model each of these as a **class** (the blueprint) and then create specific **instances** (objects) from that blueprint.

### The Four Pillars of OOP

| Pillar | Description | Neuroscience Analogy |
|---|---|---|
| **Encapsulation** | Bundle data + methods together; control access | A `Trial` object keeps its raw data private and exposes a clean `.get_filtered_signal()` method |
| **Inheritance** | Create new classes from existing ones | A `ReachingTrial` inherits from a generic `Trial` class |
| **Abstraction** | Hide complexity, expose simple interfaces | You call `trial.compute_reaction_time()` without knowing the algorithm details |
| **Polymorphism** | Different classes share the same interface | Both `EMGSignal` and `KinematicSignal` have a `.plot()` method, but each plots differently |

---
## 2. Why OOP Matters in Movement Neuroscience <a id='2'></a>

Without OOP, you might store participant data like this:

In [ ]:
# ---- The "old way" — using plain lists/dicts ----
# Each participant is a list: [name, age, handedness, injury_history]
participant_01 = ["Alice", 27, "right", "none"]
participant_02 = ["Bob", 31, "left", "ACL tear 2019"]
participant_03 = ["Carol", 24, "right"]  # Oops — forgot injury history!

# What does participant_01[2] mean again? Was index 2 the age or handedness?
print(f"Participant 1 handedness: {participant_01[2]}")
print(f"Participant 3 injury: {participant_03[2]}")  # Bug! This is handedness, not injury.

**Problems with the list approach:**
1. **Readability** — `participant_01[2]` tells you nothing about what that value represents.
2. **No guarantees** — Nothing prevents you from forgetting a field (like `participant_03` above).
3. **No built-in behaviors** — Where do you put the function that computes a participant's mean reaction time?

OOP solves all three of these. Let's see how.

---
## 3. Defining a Class in Python <a id='3'></a>

A **class** is a blueprint for creating objects. You define a class with the `class` keyword, followed by the class name (in `CamelCase` by convention) and a colon.

The `__init__` method (called the **constructor** or **initializer**) runs automatically when you create a new object. It sets up the object's initial state.

In [ ]:
# ---- Our first class: a minimal Participant ----

class Participant:
    """Represents a participant in a movement neuroscience experiment."""
    
    def __init__(self, name, age, handedness):
        # 'self' refers to the specific object being created.
        # These are INSTANCE ATTRIBUTES — unique to each participant.
        self.name = name
        self.age = age
        self.handedness = handedness

# That's it! We've defined the blueprint.
# Note: no actual participant exists yet — we've only described what one looks like.

### Key Points

- **`class Participant:`** — declares a new class called `Participant`.
- **`def __init__(self, name, age, handedness):`** — the initializer. `self` is always the first parameter; Python passes it automatically.
- **`self.name = name`** — creates an **instance attribute** called `name` on the object, assigned from the parameter `name`.
- The triple-quoted string right after the class declaration is a **docstring** — a best practice for documenting what your class does.

---
## 4. Instantiating a Class — Creating Objects <a id='4'></a>

**Instantiation** means creating a concrete object from a class blueprint. You do this by calling the class name like a function, passing in the required arguments.

In [ ]:
# ---- Create (instantiate) two Participant objects ----

p1 = Participant("Alice", 27, "right")
p2 = Participant("Bob", 31, "left")

# Access their attributes using dot notation
print(f"Participant 1: {p1.name}, age {p1.age}, {p1.handedness}-handed")
print(f"Participant 2: {p2.name}, age {p2.age}, {p2.handedness}-handed")

In [ ]:
# ---- Each object is independent ----

p1.age = 28  # Alice had a birthday!
print(f"Alice is now {p1.age}, Bob is still {p2.age}")

In [ ]:
# ---- Objects are unique, even if they have the same data ----

p3 = Participant("Alice", 28, "right")
print(f"p1 == p3? {p1 == p3}")  # False — different objects in memory
print(f"p1 is p3? {p1 is p3}")  # False — different memory addresses

In [ ]:
# ---- What happens if you forget a required argument? ----
# Uncomment the line below to see the error:

# p_bad = Participant("Carol", 24)  # TypeError: missing 'handedness'

Notice how the class **enforces structure**: you *must* provide name, age, and handedness. No more accidentally missing fields!

---
## 5. Class vs Instance Attributes <a id='5'></a>

- **Instance attributes** are defined in `__init__` using `self.xxx = ...`. They are unique to each object.
- **Class attributes** are defined directly inside the class body (outside any method). They are shared across *all* instances.

In [ ]:
class Participant:
    """A participant in a movement neuroscience experiment."""
    
    # CLASS ATTRIBUTES — same for every participant
    species = "Homo sapiens"
    study_protocol = "Reaching-Under-Perturbation v2.1"
    
    def __init__(self, name, age, handedness, injury_history="none"):
        # INSTANCE ATTRIBUTES — unique to each participant
        self.name = name
        self.age = age
        self.handedness = handedness
        self.injury_history = injury_history

p1 = Participant("Alice", 27, "right")
p2 = Participant("Bob", 31, "left", injury_history="ACL tear 2019")

# Class attributes — same for both:
print(f"p1 protocol: {p1.study_protocol}")
print(f"p2 protocol: {p2.study_protocol}")

# Instance attributes — different:
print(f"\np1 injury: {p1.injury_history}")
print(f"p2 injury: {p2.injury_history}")

| Use **class attributes** for... | Use **instance attributes** for... |
|---|---|
| Values shared by all instances | Values unique to each instance |
| Constants or defaults | Data that varies between objects |
| Example: `species`, `study_protocol` | Example: `name`, `age`, `reaction_times` |

---
## 6. Instance Methods — Adding Behavior <a id='6'></a>

Classes aren't just for storing data — they can also define **behaviors** (functions that operate on the object's data). An instance method always takes `self` as its first parameter.

In [ ]:
import numpy as np

class Participant:
    """A participant in a movement neuroscience experiment."""
    
    study_protocol = "Reaching-Under-Perturbation v2.1"
    
    def __init__(self, name, age, handedness, injury_history="none"):
        self.name = name
        self.age = age
        self.handedness = handedness
        self.injury_history = injury_history
        self.reaction_times = []  # Will store RTs (in seconds) across trials
    
    def add_reaction_time(self, rt):
        """Record a single reaction time (seconds) from a trial."""
        self.reaction_times.append(rt)
    
    def mean_reaction_time(self):
        """Compute the mean reaction time across all recorded trials."""
        if not self.reaction_times:
            return None
        return np.mean(self.reaction_times)
    
    def summary(self):
        """Return a formatted summary string for this participant."""
        mean_rt = self.mean_reaction_time()
        rt_str = f"{mean_rt:.3f} s" if mean_rt is not None else "no data"
        return (
            f"Participant: {self.name} | Age: {self.age} | "
            f"Hand: {self.handedness} | Mean RT: {rt_str}"
        )

In [ ]:
# ---- Using instance methods ----

alice = Participant("Alice", 27, "right")
for rt in [0.342, 0.298, 0.315, 0.287, 0.331]:
    alice.add_reaction_time(rt)

print(alice.summary())
print(f"All RTs: {alice.reaction_times}")

**Key insight:** The methods are *bundled with the data they operate on*. You don't need to pass the participant's data as separate arguments — the method already has access through `self`.

---
## 7. Dunder Methods — `__init__` and `__str__` <a id='7'></a>

Python has many **dunder (double-underscore) methods** that let you customize how your objects behave with built-in operations. The most important one after `__init__` is `__str__`, which controls what `print()` displays.

In [ ]:
# ---- Without __str__: unhelpful memory address ----

class ParticipantBasic:
    def __init__(self, name, age):
        self.name = name
        self.age = age

p = ParticipantBasic("Alice", 27)
print(p)  # Something like: <__main__.ParticipantBasic object at 0x7f...>

In [ ]:
# ---- With __str__: clean, informative output ----

class Participant:
    study_protocol = "Reaching-Under-Perturbation v2.1"
    
    def __init__(self, name, age, handedness, injury_history="none"):
        self.name = name
        self.age = age
        self.handedness = handedness
        self.injury_history = injury_history
        self.reaction_times = []
    
    def __str__(self):
        return f"Participant('{self.name}', age={self.age}, hand='{self.handedness}')"
    
    def add_reaction_time(self, rt):
        self.reaction_times.append(rt)
    
    def mean_reaction_time(self):
        if not self.reaction_times:
            return None
        return np.mean(self.reaction_times)

alice = Participant("Alice", 27, "right")
bob = Participant("Bob", 31, "left")
print(alice)
print(bob)

### Other useful dunder methods (for reference)

| Method | Called by | Example use case |
|---|---|---|
| `__init__(self, ...)` | Creating a new object | Set up attributes |
| `__str__(self)` | `print(obj)`, `str(obj)` | Human-readable display |
| `__repr__(self)` | `repr(obj)`, interactive shell | Unambiguous, developer-facing display |
| `__len__(self)` | `len(obj)` | Number of trials, number of channels |
| `__eq__(self, other)` | `obj1 == obj2` | Compare two participants or trials |
| `__getitem__(self, idx)` | `obj[idx]` | Index into trial data like a list |

---
# Part II — Inheritance and Polymorphism

## 8. Inheritance — Building Class Hierarchies <a id='8'></a>

**Inheritance** lets you create a new class (a **child/subclass**) that automatically gets all the attributes and methods of an existing class (the **parent/superclass**).

```python
class ChildClass(ParentClass):
    pass  # Inherits everything from ParentClass
```

In [ ]:
# ---- Parent class: a generic experimental Trial ----

class Trial:
    """A generic trial in a movement neuroscience experiment."""
    default_sampling_rate = 1000
    
    def __init__(self, trial_id, condition, duration_s):
        self.trial_id = trial_id
        self.condition = condition
        self.duration_s = duration_s
    
    def __str__(self):
        return f"Trial {self.trial_id} ({self.condition}, {self.duration_s}s)"
    
    def describe(self):
        return f"Trial {self.trial_id}: condition='{self.condition}', duration={self.duration_s}s"


# ---- Child classes: specialized trial types ----

class ReachingTrial(Trial):
    pass

class GraspingTrial(Trial):
    pass

class BalanceTrial(Trial):
    pass

In [ ]:
# ---- Child classes inherit everything from the parent ----

reach = ReachingTrial(trial_id=1, condition="perturbation", duration_s=5.0)
grasp = GraspingTrial(trial_id=2, condition="control", duration_s=3.0)
balance = BalanceTrial(trial_id=3, condition="eyes_closed", duration_s=30.0)

print(reach)             # Uses __str__ from Trial
print(grasp.describe())  # Uses describe() from Trial

# type() and isinstance():
print(f"\ntype(reach): {type(reach)}")
print(f"isinstance(reach, Trial): {isinstance(reach, Trial)}")          # True
print(f"isinstance(reach, BalanceTrial): {isinstance(reach, BalanceTrial)}")  # False

---
## 9. Overriding and Extending Parent Methods <a id='9'></a>

Child classes can **override** (replace) or **extend** (add to) parent methods using `super()`.

In [ ]:
# ---- Overriding: each trial type computes its own primary metric ----

class Trial:
    def __init__(self, trial_id, condition, duration_s):
        self.trial_id = trial_id
        self.condition = condition
        self.duration_s = duration_s
    
    def __str__(self):
        return f"Trial {self.trial_id} ({self.condition})"
    
    def primary_metric(self):
        return "No metric defined for generic Trial"


class ReachingTrial(Trial):
    def __init__(self, trial_id, condition, duration_s, reaction_time_s, endpoint_error_cm):
        super().__init__(trial_id, condition, duration_s)  # Call parent __init__
        self.reaction_time_s = reaction_time_s
        self.endpoint_error_cm = endpoint_error_cm
    
    def primary_metric(self):  # Override
        return f"Endpoint error: {self.endpoint_error_cm:.2f} cm"


class BalanceTrial(Trial):
    def __init__(self, trial_id, condition, duration_s, sway_area_cm2):
        super().__init__(trial_id, condition, duration_s)
        self.sway_area_cm2 = sway_area_cm2
    
    def primary_metric(self):  # Override
        return f"Sway area: {self.sway_area_cm2:.2f} cm²"


# ---- Polymorphism: same method name, different behavior ----
trials = [
    ReachingTrial(1, "control", 5.0, reaction_time_s=0.312, endpoint_error_cm=1.45),
    ReachingTrial(2, "perturbation", 5.0, reaction_time_s=0.389, endpoint_error_cm=3.21),
    BalanceTrial(3, "eyes_open", 30.0, sway_area_cm2=4.82),
    BalanceTrial(4, "eyes_closed", 30.0, sway_area_cm2=9.15),
]

print("=== Trial Results ===")
for t in trials:
    print(f"  {t} -> {t.primary_metric()}")

### Understanding `super()`

```python
super().__init__(trial_id, condition, duration_s)
```

This calls the **parent class's** `__init__`, so we don't re-write `self.trial_id = trial_id`, etc. We only add child-specific attributes after that call.

**Rule of thumb:** Use `super()` when you want to *extend* the parent's behavior. Skip it only when you want to *completely replace* it.

In [ ]:
# ---- Extending __str__ with super() ----

class ReachingTrialV2(Trial):
    def __init__(self, trial_id, condition, duration_s, reaction_time_s, endpoint_error_cm):
        super().__init__(trial_id, condition, duration_s)
        self.reaction_time_s = reaction_time_s
        self.endpoint_error_cm = endpoint_error_cm
    
    def __str__(self):
        base_str = super().__str__()  # Get parent's string
        return f"{base_str} | RT={self.reaction_time_s:.3f}s, Error={self.endpoint_error_cm:.2f}cm"

rt = ReachingTrialV2(1, "perturbation", 5.0, 0.389, 3.21)
print(rt)  # Combines parent + child info

---
# Part III — Properties, Encapsulation, and Composition

## 10. The `@property` Decorator — Computed Attributes <a id='10'></a>

Sometimes you want an attribute that is **computed from other data** rather than stored directly. For example, the duration of a signal can always be derived from `len(data) / sampling_rate` — you shouldn't have to store it separately (and risk it getting out of sync).

The `@property` decorator lets you define a method that **behaves like an attribute** — you access it without parentheses, but it computes its value on the fly.

In [ ]:
# ---- @property: computed attributes that auto-update ----

class Signal:
    """A time-series signal with computed properties."""
    
    def __init__(self, data, sampling_rate, label="Signal"):
        self.data = np.array(data, dtype=float)
        self.sampling_rate = sampling_rate
        self.label = label
    
    @property
    def duration(self):
        """Signal duration in seconds. Always up-to-date."""
        return len(self.data) / self.sampling_rate
    
    @property
    def time_axis(self):
        """Time vector in seconds."""
        return np.arange(len(self.data)) / self.sampling_rate
    
    @property
    def n_samples(self):
        """Number of samples."""
        return len(self.data)


# Usage — note: NO parentheses! Accessed like an attribute.
sig = Signal(np.random.randn(2000), sampling_rate=1000, label="Test")
print(f"Duration: {sig.duration} s")      # 2.0 — computed, not stored
print(f"Samples:  {sig.n_samples}")        # 2000
print(f"Time axis: {sig.time_axis[:5]}...")  # [0.000, 0.001, 0.002, ...]

In [ ]:
# ---- The property auto-updates when the underlying data changes ----

print(f"Before: duration = {sig.duration} s, n_samples = {sig.n_samples}")

# Trim the signal to the first 500 samples:
sig.data = sig.data[:500]

print(f"After:  duration = {sig.duration} s, n_samples = {sig.n_samples}")
# No manual update needed — the property recomputes automatically!

**Why not just use a regular method?** You could, but `signal.duration` reads more naturally than `signal.duration()` and makes it clear that duration is a property of the signal, not an action you're performing on it.

**When to use `@property`:**
- The value is **derived** from other attributes (no independent storage needed).
- You want it to **always stay in sync** with the data.
- You want a clean, attribute-like API: `obj.thing` instead of `obj.get_thing()`.

---
## 11. Encapsulation — Protecting Internal Data <a id='11'></a>

**Encapsulation** means controlling access to an object's internal data so that outside code can't accidentally break things.

Python doesn't have strict `private` keywords like Java or C++, but it uses **naming conventions**:

| Convention | Meaning | Example |
|---|---|---|
| `name` | Public — use freely | `signal.label` |
| `_name` | Protected — "internal use, don't touch unless you know what you're doing" | `signal._raw_data` |
| `__name` | Name-mangled — strongly discouraged from outside access | `signal.__secret` |

The single underscore `_` convention is by far the most commonly used. It's a **social contract**, not enforced by Python.

In [ ]:
# ---- Encapsulation with _ convention and @property ----
# Store the raw data as "protected" and expose a clean interface.

class EMGRecording:
    """An EMG recording with controlled access to the raw data."""
    
    def __init__(self, raw_data, sampling_rate, muscle_name):
        self._raw_data = np.array(raw_data, dtype=float)  # _ = "internal"
        self._sampling_rate = sampling_rate
        self.muscle_name = muscle_name                      # Public
    
    @property
    def sampling_rate(self):
        """Read-only sampling rate."""
        return self._sampling_rate
    
    @property
    def data(self):
        """Return a COPY of the raw data (protects the original)."""
        return self._raw_data.copy()
    
    @property
    def duration(self):
        return len(self._raw_data) / self._sampling_rate
    
    def get_rectified(self):
        """Return the full-wave rectified signal (doesn't modify original)."""
        return np.abs(self._raw_data)


# ---- Demonstration ----
emg = EMGRecording(np.random.randn(1000) * 0.5, 1000, "Biceps")

print(f"Muscle: {emg.muscle_name}")         # Public — fine
print(f"Sampling rate: {emg.sampling_rate}")  # Read-only via @property
print(f"Duration: {emg.duration} s")

# Get a copy of the data (safe — modifying it won't affect the original):
my_copy = emg.data
my_copy[:] = 999
print(f"\nOriginal data unchanged? {emg.data[0] != 999}")  # True!

**Why bother?** When you share code with collaborators (or your future self), encapsulation prevents accidental corruption. If someone writes `emg.data[:] = 0`, they only affect their copy, not the original recording.

---
## 12. Composition — "Has-A" Relationships <a id='12'></a>

**Inheritance** models "is-a" relationships: a `ReachingTrial` *is a* `Trial`.  
**Composition** models "has-a" relationships: an `Experiment` *has* `Participant` objects, which *have* `Trial` objects.

Composition is how you build complex, nested structures — and it's the pattern you'll use most in real research code.

```
Experiment
  ├── Participant "P01"
  │     ├── Trial 1 (control)
  │     ├── Trial 2 (perturbation)
  │     └── Trial 3 (washout)
  └── Participant "P02"
        ├── Trial 1 (control)
        └── Trial 2 (perturbation)
```

In [ ]:
# ---- Composition: Experiment has Participants, Participants have Trials ----

class Trial:
    def __init__(self, trial_id, condition, rt):
        self.trial_id = trial_id
        self.condition = condition
        self.rt = rt
    
    def __str__(self):
        return f"Trial {self.trial_id} ({self.condition}, RT={self.rt:.3f}s)"


class Participant:
    def __init__(self, pid, age, handedness):
        self.pid = pid
        self.age = age
        self.handedness = handedness
        self.trials = []              # Composition: a list of Trial objects
    
    def add_trial(self, trial):
        self.trials.append(trial)
    
    @property
    def n_trials(self):
        return len(self.trials)
    
    def mean_rt(self, condition=None):
        """Mean RT, optionally filtered by condition."""
        rts = [t.rt for t in self.trials
               if condition is None or t.condition == condition]
        return np.mean(rts) if rts else None
    
    def __str__(self):
        return f"Participant {self.pid} ({self.n_trials} trials)"


class Experiment:
    def __init__(self, name, date):
        self.name = name
        self.date = date
        self.participants = []        # Composition: a list of Participant objects
    
    def add_participant(self, participant):
        self.participants.append(participant)
    
    @property
    def total_trials(self):
        return sum(p.n_trials for p in self.participants)
    
    def summary(self):
        lines = [f"Experiment: {self.name} ({self.date})"]
        lines.append(f"Participants: {len(self.participants)}, Total trials: {self.total_trials}")
        for p in self.participants:
            lines.append(f"  {p} — mean RT: {p.mean_rt():.3f}s")
        return "\n".join(lines)
    
    def __str__(self):
        return f"Experiment('{self.name}', {len(self.participants)} participants)"

In [ ]:
# ---- Build a complete experiment using composition ----

exp = Experiment("Visuomotor Rotation", "2025-01-15")

# Create participants and add trials
rng = np.random.default_rng(42)
for pid in ["P01", "P02", "P03"]:
    p = Participant(pid, age=rng.integers(20, 35), handedness="right")
    for i, cond in enumerate(["baseline", "rotation", "washout"] * 3):
        p.add_trial(Trial(trial_id=i+1, condition=cond, rt=rng.normal(0.35, 0.05)))
    exp.add_participant(p)

# Now use the composed structure:
print(exp.summary())
print(f"\nP01 mean RT (rotation only): {exp.participants[0].mean_rt('rotation'):.3f}s")
print(f"P01 trial 1: {exp.participants[0].trials[0]}")

**Key takeaway:** Composition lets you build arbitrarily deep hierarchies. Each object manages its own data and exposes clean methods, so the top-level code stays readable.

---
# Part IV — Dataclasses and Full Framework

## 13. Dataclasses — Less Boilerplate for Data Containers <a id='13'></a>

Python 3.7+ introduced **dataclasses** — a decorator that automatically generates `__init__`, `__repr__`, `__eq__`, and more for classes that are primarily data containers.

For simple structures (participant metadata, trial settings, configuration), dataclasses eliminate repetitive boilerplate.

In [ ]:
# ---- Without dataclass: lots of boilerplate ----

class ParticipantOld:
    def __init__(self, pid, age, handedness, group):
        self.pid = pid
        self.age = age
        self.handedness = handedness
        self.group = group
    
    def __repr__(self):
        return f"ParticipantOld(pid='{self.pid}', age={self.age}, handedness='{self.handedness}', group='{self.group}')"
    
    def __eq__(self, other):
        return (self.pid == other.pid and self.age == other.age
                and self.handedness == other.handedness and self.group == other.group)

In [ ]:
# ---- With dataclass: all that boilerplate disappears ----

from dataclasses import dataclass, field
from typing import List

@dataclass
class ParticipantInfo:
    """Participant metadata — auto-generates __init__, __repr__, __eq__."""
    pid: str
    age: int
    handedness: str
    group: str = "control"          # Default value

# Usage — exactly the same as a regular class:
p1 = ParticipantInfo("P01", 24, "right")
p2 = ParticipantInfo("P01", 24, "right")
p3 = ParticipantInfo("P02", 31, "left", group="patient")

print(p1)               # Auto __repr__: ParticipantInfo(pid='P01', age=24, ...)
print(f"p1 == p2: {p1 == p2}")  # True! (auto __eq__ compares all fields)
print(f"p1 == p3: {p1 == p3}")  # False

In [ ]:
# ---- Dataclass with mutable default (use field(default_factory=...)) ----

@dataclass
class TrialConfig:
    """Configuration for a trial — stored as a dataclass."""
    condition: str
    target_distance_cm: float
    target_width_cm: float
    perturbation_deg: float = 0.0
    repetitions: int = 10

@dataclass
class ExperimentConfig:
    """Complete experiment configuration."""
    experiment_name: str
    sampling_rate_hz: int = 1000
    trial_configs: List[TrialConfig] = field(default_factory=list)  # Mutable default!

# Build an experiment config:
config = ExperimentConfig(
    experiment_name="Visuomotor Rotation",
    trial_configs=[
        TrialConfig("baseline", 15.0, 1.5),
        TrialConfig("rotation", 15.0, 1.5, perturbation_deg=30.0),
        TrialConfig("washout", 15.0, 1.5),
    ]
)
print(config)

### When to use dataclasses vs regular classes?

| Use **dataclasses** for... | Use **regular classes** for... |
|---|---|
| Pure data containers (metadata, configs, settings) | Objects with complex behavior (Signal, Trial) |
| When you want auto `__init__`, `__repr__`, `__eq__` | When you need `@property`, encapsulation, custom `__init__` |
| Replacing dictionaries with typed, structured objects | When inheritance hierarchies are involved |

Dataclasses and regular classes can be mixed freely — a regular class can *contain* dataclass instances (composition).

---
## 14. Putting It All Together — A Mini Movement Neuroscience Framework <a id='14'></a>

Let's build a small but realistic class hierarchy using everything we've learned: inheritance, `@property`, encapsulation, composition, and polymorphism.

1. **`Signal`** — generic time-series (parent class)
2. **`EMGSignal`** — electromyography (child class)
3. **`KinematicSignal`** — position/velocity (child class)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# PARENT CLASS: Generic Signal
# ============================================================

class Signal:
    """
    A generic time-series signal recorded during a movement trial.
    Uses @property for computed attributes and _ for internal data.
    """
    
    def __init__(self, data, sampling_rate, label="Signal"):
        self._data = np.array(data, dtype=float)  # Protected internal storage
        self._sampling_rate = sampling_rate
        self.label = label
    
    def __str__(self):
        return f"{self.label} ({self.n_samples} samples @ {self.sampling_rate} Hz)"
    
    def __len__(self):
        return len(self._data)
    
    # ---- Properties: computed, always in sync ----
    
    @property
    def data(self):
        """Return a copy of the data (protects original)."""
        return self._data.copy()
    
    @property
    def sampling_rate(self):
        return self._sampling_rate
    
    @property
    def time_axis(self):
        return np.arange(len(self._data)) / self._sampling_rate
    
    @property
    def duration(self):
        return len(self._data) / self._sampling_rate
    
    @property
    def n_samples(self):
        return len(self._data)
    
    # ---- Methods ----
    
    def mean(self):
        return np.mean(self._data)
    
    def std(self):
        return np.std(self._data)
    
    def plot(self, ax=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 3))
        ax.plot(self.time_axis, self._data, linewidth=0.8)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Amplitude")
        ax.set_title(self.label)
        return ax

print("Signal class defined.")

In [ ]:
# ============================================================
# CHILD CLASS 1: EMG Signal
# ============================================================

class EMGSignal(Signal):
    """
    EMG signal with rectification, RMS envelope, and custom plotting.
    """
    
    def __init__(self, data, sampling_rate, muscle_name):
        super().__init__(data, sampling_rate, label=f"EMG: {muscle_name}")
        self.muscle_name = muscle_name
    
    def rectify(self):
        return np.abs(self._data)
    
    def rms_envelope(self, window_ms=50):
        window_samples = int(window_ms / 1000 * self._sampling_rate)
        squared = self._data ** 2
        kernel = np.ones(window_samples) / window_samples
        return np.sqrt(np.convolve(squared, kernel, mode='same'))
    
    def plot(self, ax=None, show_envelope=True):
        """Override: EMG-specific visualization."""
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 3))
        ax.plot(self.time_axis, self._data, color='gray', alpha=0.5,
                linewidth=0.5, label='Raw')
        if show_envelope:
            rms = self.rms_envelope()
            ax.plot(self.time_axis, rms, color='tab:red', linewidth=1.5,
                    label='RMS envelope')
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("EMG (mV)")
        ax.set_title(self.label)
        ax.legend(loc='upper right', fontsize=8)
        return ax

print("EMGSignal class defined.")

In [ ]:
# ============================================================
# CHILD CLASS 2: Kinematic Signal
# ============================================================

class KinematicSignal(Signal):
    """
    Kinematic signal with differentiation and peak velocity detection.
    """
    
    def __init__(self, data, sampling_rate, body_part, dimension="x"):
        super().__init__(data, sampling_rate, label=f"Kinematics: {body_part} ({dimension})")
        self.body_part = body_part
        self.dimension = dimension
    
    def differentiate(self):
        return np.gradient(self._data, 1.0 / self._sampling_rate)
    
    @property
    def peak_velocity(self):
        """Peak absolute velocity (computed property)."""
        return np.max(np.abs(self.differentiate()))
    
    def plot(self, ax=None):
        """Override: kinematics-specific visualization."""
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 3))
        ax.plot(self.time_axis, self._data, color='tab:blue', linewidth=1.2)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel(f"Position ({self.dimension}, cm)")
        ax.set_title(self.label)
        return ax

print("KinematicSignal class defined.")

In [ ]:
# ============================================================
# DEMO: Generate synthetic data and use our framework
# ============================================================

np.random.seed(42)
fs = 1000
t = np.arange(0, 2.0, 1/fs)

# Synthetic EMG: Gaussian burst at 0.8 s
activation = np.exp(-0.5 * ((t - 0.8) / 0.15) ** 2)
emg_raw = activation * np.random.randn(len(t)) * 0.5

# Synthetic kinematics: sigmoid reaching trajectory
position = 30 / (1 + np.exp(-15 * (t - 0.9)))

# Create objects
emg_biceps = EMGSignal(emg_raw, fs, "Biceps Brachii")
hand_x = KinematicSignal(position, fs, "Hand", "x")

# Use __str__, @property, and methods
print(emg_biceps)
print(hand_x)
print(f"\nEMG duration: {emg_biceps.duration:.2f} s")
print(f"Peak hand velocity: {hand_x.peak_velocity:.1f} cm/s")
print(f"isinstance(emg_biceps, Signal): {isinstance(emg_biceps, Signal)}")

In [ ]:
# ---- Polymorphism: same .plot() call, different visualizations ----

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
hand_x.plot(ax=axes[0])
emg_biceps.plot(ax=axes[1])
axes[0].axvline(0.8, color='green', linestyle='--', alpha=0.6, label='Movement onset')
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

---
# Part V — Practice

## 15. Exercises <a id='15'></a>

---

### Exercise 1: Create a `ForceSignal` class

Create a `ForceSignal` class that inherits from `Signal`. It should:
- Accept `data`, `sampling_rate`, and `force_direction` (e.g., `"vertical"`) in `__init__`.
- Have a `@property` called `peak_force` that returns the maximum absolute force value.
- Override `plot()` to label the y-axis as `"Force (N)"`.

In [ ]:
# ---- YOUR CODE HERE ----

class ForceSignal(Signal):
    """A ground reaction force signal."""
    
    def __init__(self, data, sampling_rate, force_direction):
        # TODO: call super().__init__ with an appropriate label
        # TODO: store force_direction
        pass
    
    @property
    def peak_force(self):
        # TODO: return max absolute value
        pass
    
    def plot(self, ax=None):
        # TODO: override to set y-label to "Force (N)"
        pass

# Test:
# fake_force = np.sin(np.linspace(0, 4*np.pi, 1000)) * 450 + 700
# grf = ForceSignal(fake_force, 1000, "vertical")
# print(grf)
# print(f"Peak force: {grf.peak_force:.1f} N")
# grf.plot()
# plt.show()

### Exercise 2: Build an Experiment with Composition

Using the `Trial`, `Participant`, and `Experiment` classes from Section 12:
1. Create an `Experiment` with 2 participants.
2. Add 5 trials each (mix of "baseline" and "rotation" conditions with random RTs).
3. Print the experiment summary.
4. Compute each participant's mean RT for the "rotation" condition only.

In [ ]:
# ---- YOUR CODE HERE ----


### Exercise 3: Dataclass for Trial Metadata

Create a `@dataclass` called `TrialResult` with fields:
- `trial_id: int`
- `condition: str`
- `rt: float`
- `endpoint_error: float`
- `success: bool = True`

Then:
1. Create a list of 5 `TrialResult` objects.
2. Filter for only successful trials.
3. Compute the mean RT of successful trials.

In [ ]:
# ---- YOUR CODE HERE ----


### Exercise 4: Encapsulated Signal (Challenge)

Modify the `Signal` class to add a `@property` setter for `data` that:
1. Validates that the new data is a 1-D array.
2. Raises a `ValueError` if it's not.
3. Converts it to float dtype.

Hint:
```python
@data.setter
def data(self, new_data):
    new_data = np.array(new_data, dtype=float)
    if new_data.ndim != 1:
        raise ValueError("Data must be 1-dimensional")
    self._data = new_data
```

In [ ]:
# ---- YOUR CODE HERE ----


---
## 16. Summary & Further Reading <a id='16'></a>

### What You Learned

| Concept | What it does | Example |
|---|---|---|
| **Class** | Blueprint for objects | `class Signal:` |
| **Instance** | Concrete object | `emg = EMGSignal(data, 1000, "Biceps")` |
| **`__init__`** | Initialize state | `self.data = data` |
| **Instance attribute** | Unique per object | `self.muscle_name` |
| **Class attribute** | Shared by all instances | `default_sampling_rate = 1000` |
| **Instance method** | Function on an object | `emg.rms_envelope()` |
| **`__str__`** | Custom `print()` output | `print(emg)` → `"EMG: Biceps (2000 @ 1000 Hz)"` |
| **Inheritance** | Child inherits from parent | `class EMGSignal(Signal):` |
| **`super()`** | Call parent method | `super().__init__(data, sr, label)` |
| **Polymorphism** | Same method, different behavior | `.plot()` on EMG vs Kinematic |
| **`@property`** | Computed attributes (no parens) | `signal.duration` auto-updates |
| **Encapsulation (`_`)** | Protect internal data | `self._raw_data`, expose via `@property` |
| **Composition** | "Has-a" relationships | `Experiment` has `Participant` objects |
| **`@dataclass`** | Auto `__init__`/`__repr__`/`__eq__` | `@dataclass class ParticipantInfo:` |

### Further Reading

- [Real Python — OOP in Python 3](https://realpython.com/python3-object-oriented-programming/) (the source tutorial)
- [Real Python — @property](https://realpython.com/python-property/) — deep dive
- [Real Python — Dataclasses](https://realpython.com/python-data-classes/) — complete guide
- [Real Python — Inheritance and Composition](https://realpython.com/inheritance-composition-python/)
- [Python Official Docs — Classes](https://docs.python.org/3/tutorial/classes.html)
- [MNE-Python](https://mne.tools/) — `Raw`, `Epochs`, `Evoked` classes as real-world OOP examples

### How this maps to real research tools

| Library | OOP pattern you learned |
|---|---|
| **MNE-Python** | `mne.io.Raw` is a class with `.filter()`, `.plot()` methods |
| **OpenSim** | `Model`, `Body`, `Joint`, `Muscle` — deep composition hierarchy |
| **scikit-learn** | `LinearRegression()` — call `.fit()` and `.predict()` |
| **PyTorch** | Network layers are classes with `.forward()` — inheritance + composition |

---

*Happy coding, and may your reaction times be fast and your endpoint errors small!*